# Production of artificial data for analysis

The aim is to introduce biases to illustrate the attractions between variables

## Data produced with Llama3

Data generated using following prompt in [Huggingface Chat](https://huggingface.co/chat), using Llama-3.3-70B-Instruct.
The prompt on March 7th, 2025 was:

"Create a new spreadsheet in CSV format with 6 columns: id, age, gender, monthly_revenue, newspaper_reading_frequency, political_orientation, with 1400 rows as a representative sample of a larger population. Age should be evenly distributed. Gender values: Male, Female, Other. Gender should be 10% for Other and Male and Female evenly distributed. Political orientation: Populist, rightwing, center, leftwing, ecologist. The ecologist and populist are younger, the center older. The younger earn monthly significantly less that the older. ecologists and populist and female/other earn less than others. rightwing earn significantly more than all others. Newpaper_reading values: rarely, sometimes, week-end, 2-3 days a week, daily. The more you read, the more you earn, you are older and you are notably leftwing"

To increase the bias, the code was then manually modified.





Following prompt was added:

"add juste the code specific to this variable: Add a new variable to the table regarding time spent on social media (variable name time_social_media) evenly but following this schema
Age[18-24,25-34,35-44,45-54,55-64,65+]
Total minutes[186,140,127,130,128,102]"

and the variable added to the code. 

The values come from the [Statista website](https://www.statista.com/statistics/1484565/time-spent-social-media-us-by-age/).

Cf. also this data https://prioridata.com/data/social-media-usage/


In [1]:
import pandas as pd
import numpy as np
import random

## The 'random issue'

random.seed(42)  

The reason why 42 is chosen: "an inside joke in the scientific and science fiction community", https://www.kaggle.com/discussions/general/336747


### Another number is possible


[From Stackoverflow](https://stackoverflow.com/questions/22639587/what-does-random-seed-do-in-python) 


A random number is generated by some operation on previous value.

If there is no previous value then the current time is taken as previous value automatically. We can provide this previous value by own using random.seed(x) where x could be any number or string etc.

Hence random.random() is not actually perfect random number, it could be predicted via random.seed(x).

Hence, generating a random number is not actually random, because it runs on algorithms. Algorithms always give the same output based on the same input. This means, it depends on the value of the seed. So, in order to make it more random, time is automatically assigned to seed().


In [2]:

random.seed(45)            #seed=45  
print(random.random())         #1st rand value=0.2718754143840908
print(random.random())          #2nd rand value=0.48802820785090784
print(random.random())          #2nd rand value=0.08187934237116656

random.seed(45)          # again reasign seed=45  
print(random.random())                 #matching with 1st rand value  
print(random.random())                #matching with 2nd rand value
print(random.random())          #matching with 3rd rand value



0.2718754143840908
0.48802820785090784
0.08187934237116656
0.2718754143840908
0.48802820785090784
0.08187934237116656


In [3]:
### Sum of probalilities = 1
0.05+0.20+0.25+0.15+0.15+0.12+0.08

1.0

## The implemented solution

In [15]:


# Set the seed for reproducibility
np.random.seed(42)

# Define the number of rows
n_rows = 1700


ages = np.arange(18, 91)
probabilities = np.zeros(len(ages))

### Probability values:
## the first number, decimal, is the probability of this class
# the total number sum of propabilities must be 1
## the secont number is the number of individuals in the class
# in relation to the extension of the class : 18 to 25 (excluded) = 7
for i, age in enumerate(ages):
    if age < 25:
        probabilities[i] = 0.05 / 7
    elif age < 35:
        probabilities[i] = 0.2 / 10
    elif age < 45:
        probabilities[i] = 0.25 / 10
    elif age < 55:
        probabilities[i] = 0.15 / 10
    elif age < 65:
        probabilities[i] = 0.15 / 10
    elif age < 75:
        probabilities[i] = 0.12 / 10
    else:
        probabilities[i] = 0.08 / 16

ages = np.random.choice(ages, size=1700, p=probabilities)


age = np.round(ages)

# print(len(age))

# Generate gender (10% Other, Male and Female evenly distributed)
gender = np.random.choice(['Male', 'Female', 'Other'], n_rows, p=[0.45, 0.50, 0.05])

# Generate political orientation (with age bias)
political_orientation = []
for a in age:
    if a < 30:
        if random.random() < 0.7:
            political_orientation.append(np.random.choice(['Populist', 'Ecologist']))
        else:
            political_orientation.append(np.random.choice(['Leftwing', 'Rightwing']))
    elif a < 60:
        if random.random() < 0.3:
            political_orientation.append(np.random.choice(['Leftwing']))
        else:
            political_orientation.append(np.random.choice(['Populist', 'Ecologist', 'Center', 'Rightwing']))
    else:
        if random.random() < 0.7:
            political_orientation.append(np.random.choice(['Center', 'Populist']))
        else:
            political_orientation.append(np.random.choice(['Leftwing', 'Rightwing']))

# Generate monthly revenue (with age, political orientation, and gender bias)
monthly_revenue = []
for a, p, g in zip(age, political_orientation, gender):
    if p == 'Rightwing':
        revenue = np.random.uniform(12000, 50000)
    elif a < 30:
        if p in ['Populist', 'Ecologist']:
            revenue = np.random.uniform(1500, 4000)
        else:
            revenue = np.random.uniform(2000, 6000)
    elif a < 50:
        if p in ['Populist', 'Ecologist']:
            revenue = np.random.uniform(3000, 8000)
        else:
            revenue = np.random.uniform(7000, 15000)
    else:
        if p in ['Center']:
            revenue = np.random.uniform(7000, 30000)
        else:
            revenue = np.random.uniform(3000, 9000)

    # Adjust revenue based on gender
    if g in ['Female', 'Other']:
        revenue *= 0.8

    monthly_revenue.append(int(revenue))

# Generate newspaper reading frequency (with age, political orientation, and revenue bias)
newspaper_reading_frequency = []
for a, p, r in zip(age, political_orientation, monthly_revenue):
    if r < 3000:
        newspaper_reading_frequency.append(np.random.choice(['rarely','sometimes'], p=[0.7, 0.3]))
    elif r < 6000:
        newspaper_reading_frequency.append(np.random.choice(['rarely', 'sometimes', 'week-end'], p=[0.3, 0.2, 0.5]))
    elif r < 10000:
        if p == 'Leftwing':
            newspaper_reading_frequency.append(np.random.choice(['week-end', '3-4 days a week', 'daily'], p=[0.2, 0.5, 0.3]))
        elif p == 'Populist':
            newspaper_reading_frequency.append(np.random.choice(['week-end', 'rarely', 'sometimes'], p=[0.2, 0.5, 0.3]))
        else:
            newspaper_reading_frequency.append(np.random.choice(['week-end', 'sometimes'], p=[0.5, 0.5]))
    else:
        if p == 'Leftwing':
            newspaper_reading_frequency.append(np.random.choice(['3-4 days a week', 'daily'], p=[0.5, 0.5]))
        else:
            newspaper_reading_frequency.append(np.random.choice(['3-4 days a week', 'daily'], p=[0.7, 0.3]))



# Generate time spent on social media (based on age)
time_social_media = []
for a in age:
    if a < 25:
        time_social_media.append(np.random.normal(186, 30))  # 18-24 years old, mean 186 minutes, std dev 30 minutes
    elif a < 35:
        time_social_media.append(np.random.normal(140, 25))  # 25-34 years old, mean 140 minutes, std dev 25 minutes
    elif a < 45:
        time_social_media.append(np.random.normal(100, 20))  # 35-44 years old, mean 100 inistead of 127 minutes, std dev 20 minutes
    elif a < 55:
        time_social_media.append(np.random.normal(130, 20))  # 45-54 years old, mean 130 minutes, std dev 20 minutes
    elif a < 65:
        time_social_media.append(np.random.normal(128, 20))  # 55-64 years old, mean 128 minutes, std dev 20 minutes
    else:
        time_social_media.append(np.random.normal(7, 15))  # 65+ years old, mean 10 instead of 102 minutes, std dev 15 minutes)

# Ensure time spent on social media is non-negative
time_social_media = [max(0, int(t)) for t in time_social_media]


# Generate id
id = range(1, n_rows + 1)

# Create a pandas DataFrame
df = pd.DataFrame({
    'id': id,
    'age': age,
    'gender': gender,
   'monthly_revenue': monthly_revenue,
    'newspaper_reading_frequency': newspaper_reading_frequency,
    'political_orientation': political_orientation,
    'time_social_media': time_social_media
})


# Save to CSV
df.to_csv('data/llama3_20250315.csv', index=False)

In [16]:
df.describe()

,id,age,monthly_revenue,time_social_media
count,1700.000000,1700.000000,1700.000000,1700.000000
mean,850.500000,48.159412,10376.225882,102.088235
std,490.892045,17.660652,9159.849712,55.045344
min,1.000000,18.000000,1218.000000,0.000000
25%,425.750000,34.000000,4082.250000,77.750000
50%,850.500000,45.000000,7107.500000,114.000000
75%,1275.250000,61.000000,12683.750000,139.000000
max,1700.000000,90.000000,49884.000000,257.000000


## Approche précédente

In [ ]:



# Set the seed for reproducibility
np.random.seed(42)

# Define the number of rows
n_rows = 1700

# Generate age (evenly distributed between 18 and 80)
age = np.random.uniform(18, 80, n_rows)
#age = np.round(age / 5) * 5  # Round to nearest 5-year interval
age = age.astype(int)

# Generate gender (10% Other, Male and Female evenly distributed)
gender = np.random.choice(['Male', 'Female', 'Other'], n_rows, p=[0.45, 0.45, 0.1])

# Generate political orientation (with age bias)
political_orientation = []
for a in age:
    if a < 30:
        if random.random() < 0.7:
            political_orientation.append(np.random.choice(['Populist', 'Ecologist']))
        else:
            political_orientation.append(np.random.choice(['Leftwing', 'Rightwing']))
    elif a < 50:
        if random.random() < 0.4:
            political_orientation.append(np.random.choice(['Leftwing']))
        else:
            political_orientation.append(np.random.choice(['Populist', 'Ecologist', 'Center', 'Rightwing']))
    else:
        if random.random() < 0.55:
            political_orientation.append(np.random.choice(['Center', 'Populist']))
        else:
            political_orientation.append(np.random.choice(['Leftwing', 'Rightwing']))

# Generate monthly revenue (with age, political orientation, and gender bias)
monthly_revenue = []
for a, p, g in zip(age, political_orientation, gender):
    if p == 'Rightwing':
        revenue = np.random.uniform(12000, 50000)
    elif a < 30:
        if p in ['Populist', 'Ecologist']:
            revenue = np.random.uniform(1500, 4000)
        else:
            revenue = np.random.uniform(2000, 6000)
    elif a < 50:
        if p in ['Populist', 'Ecologist']:
            revenue = np.random.uniform(3000, 8000)
        else:
            revenue = np.random.uniform(7000, 15000)
    else:
        if p in ['Center']:
            revenue = np.random.uniform(7000, 30000)
        else:
            revenue = np.random.uniform(3000, 9000)

    # Adjust revenue based on gender
    if g in ['Female', 'Other']:
        revenue *= 0.8

    monthly_revenue.append(int(revenue))

# Generate newspaper reading frequency (with age, political orientation, and revenue bias)
newspaper_reading_frequency = []
for a, p, r in zip(age, political_orientation, monthly_revenue):
    if r < 3000:
        newspaper_reading_frequency.append(np.random.choice(['rarely','sometimes'], p=[0.7, 0.3]))
    elif r < 6000:
        newspaper_reading_frequency.append(np.random.choice(['sometimes', 'week-end'], p=[0.5, 0.5]))
    elif r < 10000:
        if p == 'Leftwing':
            newspaper_reading_frequency.append(np.random.choice(['week-end', '2-3 days a week', 'daily'], p=[0.2, 0.5, 0.3]))
        else:
            newspaper_reading_frequency.append(np.random.choice(['week-end', '2-3 days a week'], p=[0.5, 0.5]))
    else:
        if p == 'Leftwing':
            newspaper_reading_frequency.append(np.random.choice(['2-3 days a week', 'daily'], p=[0.3, 0.7]))
        else:
            newspaper_reading_frequency.append(np.random.choice(['2-3 days a week', 'daily'], p=[0.5, 0.5]))



# Generate time spent on social media (based on age)
time_social_media = []
for a in age:
    if a < 25:
        time_social_media.append(np.random.normal(186, 30))  # 18-24 years old, mean 186 minutes, std dev 30 minutes
    elif a < 35:
        time_social_media.append(np.random.normal(140, 25))  # 25-34 years old, mean 140 minutes, std dev 25 minutes
    elif a < 45:
        time_social_media.append(np.random.normal(100, 20))  # 35-44 years old, mean 100 inistead of 127 minutes, std dev 20 minutes
    elif a < 55:
        time_social_media.append(np.random.normal(130, 20))  # 45-54 years old, mean 130 minutes, std dev 20 minutes
    elif a < 65:
        time_social_media.append(np.random.normal(128, 20))  # 55-64 years old, mean 128 minutes, std dev 20 minutes
    else:
        time_social_media.append(np.random.normal(10, 15))  # 65+ years old, mean 10 instead of 102 minutes, std dev 15 minutes)

# Ensure time spent on social media is non-negative
time_social_media = [max(0, int(t)) for t in time_social_media]


# Generate id
id = range(1, n_rows + 1)

# Create a pandas DataFrame
df = pd.DataFrame({
    'id': id,
    'age': age,
    'gender': gender,
   'monthly_revenue': monthly_revenue,
    'newspaper_reading_frequency': newspaper_reading_frequency,
    'political_orientation': political_orientation,
    'time_social_media': time_social_media
})


# Save to CSV
df.to_csv('data/llama3_20250307.csv', index=False)

In [87]:
df.describe()

,id,age,monthly_revenue,time_social_media
count,1700.000000,1700.000000,1700.000000,1700.000000
mean,850.500000,48.468824,10878.267059,103.925882
std,490.892045,18.175771,10045.436213,60.758811
min,1.000000,18.000000,1220.000000,0.000000
25%,425.750000,32.000000,3802.000000,57.500000
50%,850.500000,49.000000,6958.000000,118.000000
75%,1275.250000,64.000000,13848.250000,145.000000
max,1700.000000,79.000000,49905.000000,261.000000
